# Computational Efficiency Benchmark (Table 4.3)

Measures inference throughput (clips/sec), latency and parameter counts for the three deep models on the final v2 checkpoints, assessing real-time deployment feasibility for ADAS (Table 4.3).

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

In [ ]:
import os
import time
import torch
import pandas as pd
from tqdm import tqdm

# --------------------------------------------------
# Device
# --------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --------------------------------------------------
# Checkpoint paths (edit if needed)
# --------------------------------------------------
CKPT_TSM   = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2/tsm_fast/best.pt"
CKPT_GRU   = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2/cnn_gru/best.pt"
CKPT_R3D18 = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2/r3d18/best.pt"

# --------------------------------------------------
# Utility functions
# --------------------------------------------------
def load_state_into(model, ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu")

    if isinstance(ckpt, dict):
        if "model_state" in ckpt:
            sd = ckpt["model_state"]
        elif "state_dict" in ckpt:
            sd = ckpt["state_dict"]
        else:
            sd = ckpt
    else:
        sd = ckpt

    model.load_state_dict(sd, strict=True)
    return model


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def ckpt_size_mb(path):
    return os.path.getsize(path) / (1024**2)


@torch.no_grad()
def benchmark_model(model, input_shape, iters=200, warmup=50, desc="model"):
    model = model.to(device).eval()
    x = torch.randn(*input_shape, device=device)

    print(f"\n--- {desc} ---")
    print("Input shape:", input_shape)

    # --------------------
    # Warmup
    # --------------------
    for _ in tqdm(range(warmup), desc=f"{desc} warmup"):
        _ = model(x)

    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    # --------------------
    # Timed runs
    # --------------------
    t0 = time.time()
    for _ in tqdm(range(iters), desc=f"{desc} benchmark"):
        _ = model(x)

    if device.type == "cuda":
        torch.cuda.synchronize()

    t1 = time.time()

    total = t1 - t0
    ms_per_batch = (total / iters) * 1000
    batch_size = input_shape[0]
    ms_per_clip = ms_per_batch / batch_size
    clips_per_sec = 1000.0 / ms_per_clip

    peak_mem_mb = None
    if device.type == "cuda":
        peak_mem_mb = torch.cuda.max_memory_allocated() / (1024**2)

    return {
        "batch_size": batch_size,
        "input_shape": str(tuple(input_shape)),
        "ms_per_clip": ms_per_clip,
        "clips_per_sec": clips_per_sec,
        "peak_mem_mb": peak_mem_mb,
    }


# --------------------------------------------------
# Run benchmarking
# --------------------------------------------------
rows = []

# ======================
# 1️⃣ TSM-fast
# ======================
tsm = build_tsm_fast(num_classes=2)
tsm = load_state_into(tsm, CKPT_TSM)

rows.append({
    "model": "tsm_fast",
    "params_M": count_params(tsm)/1e6,
    "ckpt_MB": ckpt_size_mb(CKPT_TSM),
    **benchmark_model(
        tsm,
        input_shape=(32,3,16,112,112),
        iters=150,
        warmup=30,
        desc="TSM-fast"
    )
})


# ======================
# 2️⃣ CNN-GRU
# ======================
gru = build_cnn_gru(num_classes=2)  # ensure hidden size matches training!
gru = load_state_into(gru, CKPT_GRU)

rows.append({
    "model": "cnn_gru",
    "params_M": count_params(gru)/1e6,
    "ckpt_MB": ckpt_size_mb(CKPT_GRU),
    **benchmark_model(
        gru,
        input_shape=(32,16,3,112,112),
        iters=150,
        warmup=30,
        desc="CNN-GRU"
    )
})


# ======================
# 3️⃣ R3D18
# ======================
r3d = build_r3d18(num_classes=2)
r3d = load_state_into(r3d, CKPT_R3D18)

rows.append({
    "model": "r3d18",
    "params_M": count_params(r3d)/1e6,
    "ckpt_MB": ckpt_size_mb(CKPT_R3D18),
    **benchmark_model(
        r3d,
        input_shape=(16,3,16,112,112),
        iters=100,
        warmup=20,
        desc="R3D18"
    )
})


# --------------------------------------------------
# Final Table
# --------------------------------------------------
eff_df = pd.DataFrame(rows)
print("\n\n===== FINAL EFFICIENCY TABLE =====")
display(eff_df)

In [ ]:
import torchvision
import torch.nn as nn

NUM_CLASSES = 2

# 3D-CNN: 3D ResNet-18
def build_r3d18(num_classes=2):
    m = torchvision.models.video.r3d_18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

# CNN-GRU: ResNet18 per-frame + GRU
class ResNetGRU(nn.Module):
    def __init__(self, hidden=128, num_classes=2):
        super().__init__()
        base = torchvision.models.resnet18(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.feat_dim = 512
        self.gru = nn.GRU(self.feat_dim, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, num_classes)

    def forward(self, x):  # (B,T,C,H,W)
        B,T,C,H,W = x.shape
        x = x.reshape(B*T, C, H, W)
        f = self.backbone(x).flatten(1)
        f = f.reshape(B, T, self.feat_dim)
        out, _ = self.gru(f)
        return self.fc(out[:, -1, :])

def build_cnn_gru(num_classes=2):
    # ✅ MUST be 128 to match your saved checkpoint
    return ResNetGRU(hidden=128, num_classes=num_classes)

# TSM-style temporal average baseline
class ResNetTemporalAvg(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        base = torchvision.models.resnet18(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):  # (B,C,T,H,W)
        B,C,T,H,W = x.shape
        x = x.permute(0,2,1,3,4).contiguous().view(B*T, C, H, W)
        f = self.backbone(x).flatten(1).view(B, T, 512).mean(dim=1)
        return self.fc(f)

def build_tsm_fast(num_classes=2):
    return ResNetTemporalAvg(num_classes=num_classes)

print("✅ Builders defined.")
print("Sanity check hidden:", build_cnn_gru().gru.hidden_size)

In [ ]:
import os
import matplotlib.pyplot as plt

OUT_DIR = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2/final_test_eval"
os.makedirs(OUT_DIR, exist_ok=True)

# ---- Save CSV ----
csv_path = os.path.join(OUT_DIR, "efficiency_benchmark_cpu.csv")
eff_df.to_csv(csv_path, index=False)
print("✅ Saved CSV:", csv_path)

# ---- Save PNG table ----
png_path = os.path.join(OUT_DIR, "efficiency_benchmark_cpu.png")

df_show = eff_df.copy()

# pretty formatting
df_show["params_M"] = df_show["params_M"].map(lambda x: f"{x:.2f}")
df_show["ckpt_MB"] = df_show["ckpt_MB"].map(lambda x: f"{x:.1f}")
df_show["ms_per_clip"] = df_show["ms_per_clip"].map(lambda x: f"{x:.1f}")
df_show["clips_per_sec"] = df_show["clips_per_sec"].map(lambda x: f"{x:.2f}")
df_show["peak_mem_mb"] = df_show["peak_mem_mb"].map(lambda x: "-" if pd.isna(x) else f"{x:.1f}")

fig_w = 14
fig_h = 2.2 + 0.6 * len(df_show)
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.axis("off")

table = ax.table(
    cellText=df_show.values,
    colLabels=df_show.columns.tolist(),
    cellLoc="center",
    loc="center"
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)

plt.tight_layout()
plt.savefig(png_path, dpi=250, bbox_inches="tight")
plt.close(fig)

print("✅ Saved PNG:", png_path)

In [ ]:
import os, time, torch, pandas as pd
from tqdm import tqdm

# ============================================================
# Device
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# Checkpoints (edit if needed)
# ============================================================
CKPT_TSM   = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2/tsm_fast/best.pt"
CKPT_GRU   = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2/cnn_gru/best.pt"
CKPT_R3D18 = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2/r3d18/best.pt"

# ============================================================
# Output folder (Drive)
# ============================================================
OUT_DIR = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2/final_test_eval"
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# Helpers
# ============================================================
def load_state_into(model, ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if isinstance(ckpt, dict):
        if "model_state" in ckpt:
            sd = ckpt["model_state"]
        elif "state_dict" in ckpt:
            sd = ckpt["state_dict"]
        else:
            sd = ckpt
    else:
        sd = ckpt
    model.load_state_dict(sd, strict=True)
    return model

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def ckpt_size_mb(path):
    return os.path.getsize(path) / (1024**2)

@torch.no_grad()
def benchmark_model(model, input_shape, iters=200, warmup=50, desc="model"):
    model = model.to(device).eval()
    x = torch.randn(*input_shape, device=device)

    # warmup
    for _ in tqdm(range(warmup), desc=f"{desc} warmup", leave=False):
        _ = model(x)

    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    # timed
    t0 = time.time()
    for _ in tqdm(range(iters), desc=f"{desc} benchmark", leave=False):
        _ = model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t1 = time.time()

    total = t1 - t0
    ms_per_batch = (total / iters) * 1000
    B = input_shape[0]
    ms_per_clip = ms_per_batch / B
    clips_per_sec = 1000.0 / ms_per_clip

    peak_mem_mb = None
    if device.type == "cuda":
        peak_mem_mb = torch.cuda.max_memory_allocated() / (1024**2)

    return {
        "batch_size": B,
        "input_shape": str(tuple(input_shape)),
        "ms_per_clip": ms_per_clip,
        "clips_per_sec": clips_per_sec,
        "peak_mem_mb": peak_mem_mb,
    }

# ============================================================
# IMPORTANT:
# Your builders must already be defined:
#   build_tsm_fast(num_classes=2)
#   build_cnn_gru(num_classes=2)  # MUST be hidden=128 for your checkpoint
#   build_r3d18(num_classes=2)
# ============================================================

# ============================================================
# Run both BS=1 (latency) and BS=32 (throughput)
# ============================================================
bench_specs = {
    "tsm_fast": {
        "builder": lambda: build_tsm_fast(num_classes=2),
        "ckpt": CKPT_TSM,
        "shapes": [(1,3,16,112,112), (32,3,16,112,112)],
        "iters":  {1: 300, 32: 150},
        "warmup": {1: 50,  32: 30},
    },
    "cnn_gru": {
        "builder": lambda: build_cnn_gru(num_classes=2),
        "ckpt": CKPT_GRU,
        "shapes": [(1,16,3,112,112), (32,16,3,112,112)],
        "iters":  {1: 300, 32: 150},
        "warmup": {1: 50,  32: 30},
    },
    "r3d18": {
        "builder": lambda: build_r3d18(num_classes=2),
        "ckpt": CKPT_R3D18,
        # r3d18 is heavier: keep batch 16 for the "throughput" case
        "shapes": [(1,3,16,112,112), (16,3,16,112,112)],
        "iters":  {1: 200, 16: 100},
        "warmup": {1: 30,  16: 20},
    }
}

rows = []

for name, spec in bench_specs.items():
    print("\n==============================")
    print("Benchmarking:", name)
    print("==============================")

    model = spec["builder"]()
    model = load_state_into(model, spec["ckpt"])

    base = {
        "model": name,
        "params_M": count_params(model)/1e6,
        "ckpt_MB": ckpt_size_mb(spec["ckpt"]),
    }

    for shape in spec["shapes"]:
        B = shape[0]
        iters = spec["iters"][B]
        warm  = spec["warmup"][B]

        r = benchmark_model(
            model,
            input_shape=shape,
            iters=iters,
            warmup=warm,
            desc=f"{name} BS={B}"
        )
        rows.append({**base, **r})

eff_df2 = pd.DataFrame(rows)

# Nice sorted display
eff_df2 = eff_df2.sort_values(["model", "batch_size"]).reset_index(drop=True)
eff_df2

In [ ]:
import matplotlib.pyplot as plt

csv_path = os.path.join(OUT_DIR, f"efficiency_benchmark_{device.type}_bs1_bsX.csv")
eff_df2.to_csv(csv_path, index=False)
print("✅ Saved CSV:", csv_path)

# PNG table
png_path = os.path.join(OUT_DIR, f"efficiency_benchmark_{device.type}_bs1_bsX.png")

df_show = eff_df2.copy()
df_show["params_M"] = df_show["params_M"].map(lambda x: f"{x:.2f}")
df_show["ckpt_MB"] = df_show["ckpt_MB"].map(lambda x: f"{x:.1f}")
df_show["ms_per_clip"] = df_show["ms_per_clip"].map(lambda x: f"{x:.1f}")
df_show["clips_per_sec"] = df_show["clips_per_sec"].map(lambda x: f"{x:.2f}")
df_show["peak_mem_mb"] = df_show["peak_mem_mb"].map(lambda x: "-" if pd.isna(x) else f"{x:.1f}")

fig_w = 14
fig_h = 2.2 + 0.55 * len(df_show)
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.axis("off")

table = ax.table(
    cellText=df_show.values,
    colLabels=df_show.columns.tolist(),
    cellLoc="center",
    loc="center"
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.45)

plt.tight_layout()
plt.savefig(png_path, dpi=250, bbox_inches="tight")
plt.close(fig)

print("✅ Saved PNG:", png_path)